In [2]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_csv("../data/victorian_road_crash_data.csv")

# Keep the original dataset untouched
df_working = df.copy()

print("Original dataset shape :", df.shape)
print("Working copy shape     :", df_working.shape)

Original dataset shape : (200352, 52)
Working copy shape     : (200352, 52)


In [3]:
DATA_PATH = "../data/victorian_road_crash_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [4]:
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

Rows    : 200352
Columns : 52


In [5]:
df.head()

,ACCIDENT_NO,ACCIDENT_DATE,ACCIDENT_TIME,ACCIDENT_TYPE,DAY_OF_WEEK,DCA_CODE,DCA_CODE_DESCRIPTION,LIGHT_CONDITION,POLICE_ATTEND,ROAD_GEOMETRY,...,NO_OF_VEHICLES,HEAVYVEHICLE,PASSENGERVEHICLE,MOTORCYCLE,PT_VEHICLE,DEG_URBAN_NAME,SRNS,RMA,DIVIDED,STAT_DIV_NAME
0,T20140024624,27-11-2014,18:35:00,Collision with vehicle,Thursday,110,CROSS TRAFFIC(INTERSECTIONS ONLY),Day,Yes,Cross intersection,...,2.0,0.0,2.0,0.0,0.0,TOWNS,NaN,Local Road,Undivided,Country
1,T20190026336,27-12-2019,15:45:00,Collision with vehicle,Friday,113,RIGHT NEAR (INTERSECTIONS ONLY),Day,Yes,T intersection,...,2.0,0.0,2.0,0.0,0.0,MELB_URBAN,NaN,Arterial Other,Divided,Metro
2,T20190019196,02-10-2019,12:07:00,Collision with vehicle,Wednesday,173,RIGHT OFF CARRIAGEWAY INTO OBJECT/PARKED VEHICLE,Day,Yes,Not at intersection,...,3.0,0.0,2.0,0.0,0.0,MELB_URBAN,M,Freeway,Divided,Metro
3,T20250029202,07-11-2025,13:10:00,Struck Pedestrian,Friday,100,PED NEAR SIDE. PED HIT BY VEHICLE FROM THE RIGHT.,Day,Yes,Not at intersection,...,1.0,0.0,1.0,0.0,0.0,MELB_URBAN,NaN,Arterial Highway,Divided,Metro
4,T20210005363,30-01-2021,06:30:00,Collision with a fixed object,Saturday,183,OFF LEFT BEND INTO OBJECT/PARKED VEHICLE,Dusk/Dawn,No,Not at intersection,...,1.0,0.0,1.0,0.0,0.0,RURAL_VICTORIA,C,Arterial Other,Undivided,Metro


In [6]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [7]:
df = df.drop_duplicates()

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (200352, 52)


In [8]:
print(df["SEVERITY"].value_counts())

SEVERITY
Other injury accident      124942
Serious injury accident     72054
Fatal accident               3352
Non injury accident             4
Name: count, dtype: int64


In [9]:
non_injury_count = (df["SEVERITY"] == "Non injury accident").sum()

print("Non-injury accidents:", non_injury_count)

Non-injury accidents: 4


In [10]:
df = df[df["SEVERITY"] != "Non injury accident"].copy()

print("Dataset shape:", df.shape)

Dataset shape: (200348, 52)


In [11]:
X = df.drop("SEVERITY", axis=1)
y = df["SEVERITY"]
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget classes:")
print(y.value_counts())

X shape: (200348, 51)
y shape: (200348,)

Target classes:
SEVERITY
Other injury accident      124942
Serious injury accident     72054
Fatal accident               3352
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))
print("Training percentage:", len(X_train) / len(X) * 100)
print("Testing percentage :", len(X_test) / len(X) * 100)

Training rows: 160278
Testing rows : 40070
Training percentage: 79.99980034739554
Testing percentage : 20.00019965260447


In [15]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numerical features:")
print(numeric_features)
print("\nNumber of numerical features:", len(numeric_features))

print("\nCategorical features:")
print(categorical_features)
print("\nNumber of categorical features:", len(categorical_features))

Numerical features:
['DCA_CODE', 'ROAD_ROUTE_1', 'LATITUDE', 'LONGITUDE', 'VICGRID_X', 'VICGRID_Y', 'TOTAL_PERSONS', 'MALES', 'FEMALES', 'BICYCLIST', 'PASSENGER', 'DRIVER', 'PEDESTRIAN', 'PILLION', 'MOTORCYCLIST', 'UNKNOWN', 'PED_CYCLIST_5_12', 'PED_CYCLIST_13_18', 'OLD_PED_65_AND_OVER', 'OLD_DRIVER_75_AND_OVER', 'YOUNG_DRIVER_18_25', 'NO_OF_VEHICLES', 'HEAVYVEHICLE', 'PASSENGERVEHICLE', 'MOTORCYCLE', 'PT_VEHICLE']

Number of numerical features: 26

Categorical features:
['ACCIDENT_DATE', 'ACCIDENT_TIME', 'ACCIDENT_TYPE', 'DAY_OF_WEEK', 'DCA_CODE_DESCRIPTION', 'LIGHT_CONDITION', 'POLICE_ATTEND', 'ROAD_GEOMETRY', 'SPEED_ZONE', 'RUN_OFFROAD', 'ROAD_NAME', 'ROAD_TYPE', 'LGA_NAME', 'DTP_REGION', 'DEG_URBAN_NAME', 'SRNS', 'RMA', 'DIVIDED', 'STAT_DIV_NAME']

Number of categorical features: 19


## Removing Identifier and Target-Leakage Features

Based on the literature, variables that directly encode accident outcomes or serve only as identifiers should not be used as predictive features.

`ACCIDENT_NO` is a unique accident identifier and does not provide meaningful predictive information.

The variables `INJ_OR_FATAL`, `FATALITY`, `SERIOUSINJURY`, `OTHERINJURY`, and `NONINJURED` describe injury outcomes associated with the accident and may directly reveal the target `SEVERITY`. These variables are therefore excluded to prevent target leakage.

In [14]:
drop_columns = [
    "ACCIDENT_NO",
    "INJ_OR_FATAL",
    "FATALITY",
    "SERIOUSINJURY",
    "OTHERINJURY",
    "NONINJURED"
]

X_train = X_train.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

print("Removed columns:")
print(drop_columns)

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

Removed columns:
['ACCIDENT_NO', 'INJ_OR_FATAL', 'FATALITY', 'SERIOUSINJURY', 'OTHERINJURY', 'NONINJURED']

X_train shape: (160278, 45)
X_test shape : (40070, 45)


## Date and Time Feature Engineering

Accident timing can influence crash severity through variations in traffic conditions, visibility, and road usage patterns. Following the literature's use of temporal accident characteristics, the raw date and time fields are transformed into numerical temporal features.

In [16]:
def add_datetime_features(data):
    data = data.copy()

    data["ACCIDENT_DATE"] = pd.to_datetime(
        data["ACCIDENT_DATE"],
        errors="coerce"
    )

    data["ACCIDENT_YEAR"] = data["ACCIDENT_DATE"].dt.year
    data["ACCIDENT_MONTH"] = data["ACCIDENT_DATE"].dt.month
    data["ACCIDENT_DAY"] = data["ACCIDENT_DATE"].dt.day

    data["ACCIDENT_TIME"] = pd.to_datetime(
        data["ACCIDENT_TIME"],
        format="mixed",
        errors="coerce"
    )

    data["ACCIDENT_HOUR"] = data["ACCIDENT_TIME"].dt.hour
    data["ACCIDENT_MINUTE"] = data["ACCIDENT_TIME"].dt.minute

    data["IS_WEEKEND"] = data["DAY_OF_WEEK"].isin(
        ["Saturday", "Sunday"]
    ).astype(int)

    data = data.drop(
        columns=["ACCIDENT_DATE", "ACCIDENT_TIME"],
        errors="ignore"
    )

    return data

In [17]:
X_train = add_datetime_features(X_train)
X_test = add_datetime_features(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (160278, 49)
X_test shape : (40070, 49)


In [18]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numerical features: 32
['DCA_CODE', 'ROAD_ROUTE_1', 'LATITUDE', 'LONGITUDE', 'VICGRID_X', 'VICGRID_Y', 'TOTAL_PERSONS', 'MALES', 'FEMALES', 'BICYCLIST', 'PASSENGER', 'DRIVER', 'PEDESTRIAN', 'PILLION', 'MOTORCYCLIST', 'UNKNOWN', 'PED_CYCLIST_5_12', 'PED_CYCLIST_13_18', 'OLD_PED_65_AND_OVER', 'OLD_DRIVER_75_AND_OVER', 'YOUNG_DRIVER_18_25', 'NO_OF_VEHICLES', 'HEAVYVEHICLE', 'PASSENGERVEHICLE', 'MOTORCYCLE', 'PT_VEHICLE', 'ACCIDENT_YEAR', 'ACCIDENT_MONTH', 'ACCIDENT_DAY', 'ACCIDENT_HOUR', 'ACCIDENT_MINUTE', 'IS_WEEKEND']

Categorical features: 17
['ACCIDENT_TYPE', 'DAY_OF_WEEK', 'DCA_CODE_DESCRIPTION', 'LIGHT_CONDITION', 'POLICE_ATTEND', 'ROAD_GEOMETRY', 'SPEED_ZONE', 'RUN_OFFROAD', 'ROAD_NAME', 'ROAD_TYPE', 'LGA_NAME', 'DTP_REGION', 'DEG_URBAN_NAME', 'SRNS', 'RMA', 'DIVIDED', 'STAT_DIV_NAME']


## Missing Value Analysis

Missing values are handled separately for numerical and categorical features. Numerical variables will be imputed using the median, while categorical variables will be imputed using the most frequent category. The preprocessing parameters will be learned only from the training data.

In [19]:
missing_data = pd.DataFrame({
    "Missing Values": X_train.isnull().sum(),
    "Percentage": (
        X_train.isnull().sum() /
        len(X_train) * 100
    ).round(2)
})

missing_data = missing_data[
    missing_data["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

display(missing_data)

,Missing Values,Percentage
SRNS,112226,70.02
ACCIDENT_YEAR,97500,60.83
ACCIDENT_DAY,97500,60.83
ACCIDENT_MONTH,97500,60.83
DIVIDED,6152,3.84
RMA,6152,3.84
ROAD_TYPE,2200,1.37
STAT_DIV_NAME,803,0.50
DEG_URBAN_NAME,786,0.49
ROAD_NAME,214,0.13


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [22]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

In [23]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

In [24]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [25]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Target mapping:")

for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)

Target mapping:
0 -> Fatal accident
1 -> Other injury accident
2 -> Serious injury accident


In [26]:
print("Final processed shapes")
print("=" * 40)

print("X_train:", X_train_processed.shape)
print("X_test :", X_test_processed.shape)
print("y_train:", y_train_encoded.shape)
print("y_test :", y_test_encoded.shape)

Final processed shapes
X_train: (160278, 13615)
X_test : (40070, 13615)
y_train: (160278,)
y_test : (40070,)


In [27]:
print("NaN values in X_train:",
      np.isnan(X_train_processed).sum())

print("NaN values in X_test:",
      np.isnan(X_test_processed).sum())

NaN values in X_train: 0
NaN values in X_test: 0


## Saving Processed Data

In [28]:
import os

os.makedirs("../data/processed", exist_ok=True)

X_train_df = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out()
)

X_test_df = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out()
)

X_train_df.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test_df.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

pd.Series(y_train_encoded, name="SEVERITY").to_csv(
    "../data/processed/y_train.csv",
    index=False
)

pd.Series(y_test_encoded, name="SEVERITY").to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Processed datasets saved successfully.")

KeyboardInterrupt: 